In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("sci.mplstyle")

In [2]:
def tinh_he_so_forward(kappa, C, rho, N_x, N_time, L, t_max):
    h = L / (N_x - 1) # buoc luoi khong gian
    k = t_max / (N_time - 1) # buoc luoi thoi gian

    alpha = np.sqrt(kappa / (C * rho)) #  he so khuech tan nhiet

    eta = alpha**2 * k / h**2 # he so on dinh

    if eta > 0.5:
        print(f"So do Forward Euler khong on dinh: eta = {eta:.6f} > 0.5")
        print("Hay tang N_time hoac giam N_x.")
        return None

    return eta, h, k, alpha

In [3]:
def ham_ghi_file(file, dk_bai_toan): # dung de ghi thong tin quan trong ra file

    file.write(f"# {dk_bai_toan['mo_ta']}\n")
    file.write("# Forward Euler cho phuong trinh truyen nhiet 1D\n")
    file.write("# Dieu kien on dinh: eta <= 0.5\n")
    file.write("#\n")

    file.write(f"# N_x    = {dk_bai_toan['N_x']}\n")
    file.write(f"# N_time = {dk_bai_toan['N_time']}\n")
    file.write(f"# L      = {dk_bai_toan['L']}\n")
    file.write(f"# t_max  = {dk_bai_toan['t_max']}\n")
    file.write(f"# h      = {dk_bai_toan['h']:.8e}\n")
    file.write(f"# k      = {dk_bai_toan['k']:.8e}\n")
    file.write(f"# eta    = {dk_bai_toan['eta']:.8e}\n")

    file.write("#\n")
    file.write(f"# {'j (t)':>8s} {'i (x)':>8s} {'t':>15s} {'x':>15s} {'u':>15s}\n")

def ham_luu_dulieu(filename, u, dk_bai_toan, skip_buoc_thoigian):
    N_x = dk_bai_toan["N_x"]
    N_time = dk_bai_toan["N_time"]
    h = dk_bai_toan["h"]
    k = dk_bai_toan["k"]

    with open(filename, "w", encoding="utf-8") as file:
        ham_ghi_file(file, dk_bai_toan)

        for j in range(N_time):
            if j % skip_buoc_thoigian != 0 and j != N_time - 1: # khi j khong chia het cho skip_buoc_thoigian thi bo qua data do va lay luon  data cuoi cung
                continue
            t = j * k

            for i in range(N_x):
                x = i * h
                file.write(f"  {j:8d} {i:8d} {t:15.6e} {x:15.6e} {u[i, j]:15.6e}\n")

            file.write("\n")

In [4]:
def ham_tinh_forward_euler(u, kappa, C, rho, L, t_max, filename, mo_ta, skip_buoc_thoigian = 100):
    
    N_x = u.shape[0]
    N_time = u.shape[1]

    eta, h, k, alpha = tinh_he_so_forward(kappa, C, rho, N_x, N_time, L, t_max)

    # Forward Euler
    for j in range(N_time - 1):
        for i in range(1, N_x - 1): # ne 2 bien ra
            u[i, j + 1] = ( (1.0 - 2.0 * eta) * u[i, j] + eta * (u[i + 1, j] + u[i - 1, j]))
        
        # Ep 2 dieu kien bien
        u[0, j] = u[0,0]
        u[N_x-1, j] = u[N_x-1, 0]

    # du lieu de ghi file
    dk_bai_toan = {
        "mo_ta": mo_ta,
        "N_x": N_x,
        "N_time": N_time,
        "L": L,
        "t_max": t_max,
        "h": h,
        "k": k,
        "eta": eta,
    }

    ham_luu_dulieu(filename + "_forward_result.txt", u, dk_bai_toan, skip_buoc_thoigian)

    print("Da tinh xong")
    print(f"eta = {eta:.6e}")
    ten_file = filename + "_forward_result.txt"
    print(f"Da luu file: {ten_file}")

    return u, dk_bai_toan

In [5]:
# =========================================================
# Bai 1: mot thanh
# =========================================================

def tao_dieu_kien_bien_bai1(N_time, N_x, L, T_ban_dau, T_trai, T_phai):
    """
    Bai 1:
        Thanh dai L
        Ban dau: T(x, 0) = T_ban_dau
        Hai dau: T(0, t) = T_trai, T(L, t) = T_phai

    N_time    : so diem thoi gian
    N_x       : so diem khong gian
    L         : chieu dai thanh
    T_ban_dau : nhiet do ban dau cua thanh
    T_trai    : nhiet do dau trai
    T_phai    : nhiet do dau phai
    """

    x = np.linspace(0.0, L, N_x)
    u = np.zeros((N_x, N_time), dtype=float)

    # Dieu kien dau: T(x, 0) = T_ban_dau
    for i in range(N_x):
        u[i, 0] = T_ban_dau

    # Dieu kien bien trai: T(0, t) = T_trai
    for j in range(N_time):
        u[0, j] = T_trai

    # Dieu kien bien phai: T(L, t) = T_phai
    for j in range(N_time):
        u[N_x - 1, j] = T_phai

    # Tai t = 0, uu tien dieu kien bien tai hai dau
    u[0, 0] = T_trai
    u[N_x - 1, 0] = T_phai

    return u, x

N_time = 6000
N_x = 100

L = 1.0
t_max = 1800.0

T_ban_dau = 100.0
T_trai = 0.0
T_phai = 0.0

kappa = 210.0
C = 900.0
rho = 2700.0

u, x = tao_dieu_kien_bien_bai1(
    N_time=N_time,
    N_x=N_x,
    L=L,
    T_ban_dau=T_ban_dau,
    T_trai=T_trai,
    T_phai=T_phai,
)

u, info = ham_tinh_forward_euler(
    u=u,
    kappa=kappa,
    C=C,
    rho=rho,
    L=L,
    t_max=t_max,
    filename="truyen_nhiet_1thanh",
    mo_ta="Bai toan truyen nhiet 1D: mot thanh",
    skip_buoc_thoigian = 20,
)

Da tinh xong
eta = 2.541424e-01
Da luu file: truyen_nhiet_1thanh_forward_result.txt


In [6]:
# =========================================================
# Bai 2: hai thanh tiep xuc
# =========================================================

# =========================================================
# 3. Tao dieu kien bien bai 2
# =========================================================

def tao_dieu_kien_bien_bai2(
    N_time,
    N_x,
    l_bar,
    T_thanh_trai,
    T_thanh_phai,
    T_bien_trai,
    T_bien_phai,
):
    """
    Bai 2:
        Hai thanh tiep xuc
        Moi thanh dai l_bar
        Tong chieu dai L = 2*l_bar

        Thanh trai ban dau co nhiet do T_thanh_trai
        Thanh phai ban dau co nhiet do T_thanh_phai

        Hai dau ngoai cung giu nhiet do:
            T(0, t) = T_bien_trai
            T(L, t) = T_bien_phai

    N_time       : so diem thoi gian
    N_x          : so diem khong gian
    l_bar        : chieu dai moi thanh
    T_thanh_trai : nhiet do ban dau cua thanh trai
    T_thanh_phai : nhiet do ban dau cua thanh phai
    T_bien_trai  : nhiet do bien trai
    T_bien_phai  : nhiet do bien phai
    """

    L = 2.0 * l_bar
    x = np.linspace(0.0, L, N_x)
    u = np.zeros((N_x, N_time), dtype=float)

    # Dieu kien dau:
    # Neu x <= l_bar thi diem do nam tren thanh trai
    # Neu x >  l_bar thi diem do nam tren thanh phai
    for i in range(N_x):
        if x[i] <= l_bar:
            u[i, 0] = T_thanh_trai
        else:
            u[i, 0] = T_thanh_phai

    # Dieu kien bien trai
    for j in range(N_time):
        u[0, j] = T_bien_trai

    # Dieu kien bien phai
    for j in range(N_time):
        u[N_x - 1, j] = T_bien_phai

    # Tai t = 0, uu tien dieu kien bien tai hai dau
    u[0, 0] = T_bien_trai
    u[N_x - 1, 0] = T_bien_phai

    return u, x, L

N_time = 15000
N_x = 101

l_bar = 0.25
t_max = 1800.0

T_thanh_trai = 100.0
T_thanh_phai = 50.0

T_bien_trai = 0.0
T_bien_phai = 0.0

kappa = 210.0
C = 900.0
rho = 2700.0

u, x, L = tao_dieu_kien_bien_bai2(
    N_time=N_time,
    N_x=N_x,
    l_bar=l_bar,
    T_thanh_trai=T_thanh_trai,
    T_thanh_phai=T_thanh_phai,
    T_bien_trai=T_bien_trai,
    T_bien_phai=T_bien_phai,
)

u, info = ham_tinh_forward_euler(
    u=u,
    kappa=kappa,
    C=C,
    rho=rho,
    L=L,
    t_max=t_max,
    filename="truyen_nhiet_2thanh",
    mo_ta="Bai toan truyen nhiet 1D: hai thanh tiep xuc",
    skip_buoc_thoigian = 20,
)

Da tinh xong
eta = 4.148425e-01
Da luu file: truyen_nhiet_2thanh_forward_result.txt
